# Simple Linear Programming (LP) Example with PuLP

cuOpt is NVIDIA's GPU accelerated solver that delivers massive speedups for real-world LP, MIP, and VRP workloads.

cuOpt seemlessly integrates with modeling languages. You can drop cuOpt into existing models built with PuLP and AMPL, with minimal refactoring. Let's take a look at an example solving a simple LP problem with cuOpt.

This example is adapted from CVXPY. You can look at the original example [here](https://www.cvxpy.org/examples/basic/linear_program.html)

## 1. Install Dependencies

To make sure we are good to go, let's install PuLP and cuOpt.

__[PuLP](https://coin-or.github.io/pulp/)__ is a popular linear and mixed integer programming modeler written in Python.




In [3]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>
            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

True

In [4]:
#!pip install pulp==3.3.0

In [ ]:
# Enable this in case you are running this in google colab or such places where cuOpt is not yet installed
#!pip uninstall -y cuda-python cuda-bindings cuda-core
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu12 nvidia-nvjitlink-cu12 rapids-logger==0.1.19
#!pip install --upgrade --extra-index-url=https://pypi.nvidia.com cuopt-cu13 nvidia-nvjitlink-cu13 rapids-logger==0.1.19

## 2. Problem Setup

This optimization problem defines a randomly generated linear program (LP) with 10 decision variables and 15 inequality constraints. The objective is to minimize a linear function of the variables, defined by a vector c, subject to linear inequality constraints of the form Ax≤b, where the matrix A and vector b are constructed to ensure feasibility using random values.


In [6]:
# Import packages.
from pulp import *
import numpy as np

# Generate a random non-trivial linear program.
m = 15  #inequalities
n = 10  #decision variables
np.random.seed(1)
s0 = np.random.randn(m)
lamb0 = np.maximum(-s0, 0)
s0 = np.maximum(s0, 0)
x0 = np.random.randn(n)
A = np.random.randn(m, n)
b = A @ x0 + s0
c = -A.T @ lamb0


In [7]:
# Define and solve the Pulp problem
prob = LpProblem("LP_example", LpMinimize)
x = [LpVariable(f"x{i}", lowBound=None) for i in range(n)]
prob += lpSum([c[i] * x[i] for i in range(n)]), "Objective"

for i in range(m):
    prob += lpSum([A[i, j] * x[j] for j in range(n)]) <= b[i], f"Constraint_{i}"

In [8]:
print(prob)

LP_example:
MINIMIZE
5.9159384978326255*x0 + -0.3865327600465154*x1 + -1.583769082503147*x2 + -3.456149761894742*x3 + -4.001866243462928*x4 + 2.3093927208615117*x5 + 0.4855808984284459*x6 + -2.7645038322063735*x7 + -4.554282549129423*x8 + -1.0277935948594457*x9 + 0.0
SUBJECT TO
Constraint_0: - 0.683727859174 x0 - 0.122890225519 x1 - 0.935769434259 x2
 - 0.267888079626 x3 + 0.530355466738 x4 - 0.691660751725 x5
 - 0.396753526856 x6 - 0.68717270012 x7 - 0.845205641499 x8
 - 0.671246130837 x9 <= 2.17495142475

Constraint_1: - 0.0126645989189 x0 - 1.11731034864 x1 + 0.234415697817 x2
 + 1.65980217711 x3 + 0.742044160577 x4 - 0.191835552362 x5
 - 0.887628964085 x6 - 0.747158293751 x7 + 1.69245460103 x8
 + 0.050807754776 x9 <= -0.079020889802

Constraint_2: - 0.636995646569 x0 + 0.190915484667 x1 + 2.10025513648 x2
 + 0.120158952482 x3 + 0.617203109707 x4 + 0.300170319956 x5
 - 0.352249846494 x6 - 1.14251819802 x7 - 0.349342722413 x8
 - 0.208894233375 x9 <= -2.93864431564

Constraint_3: 0.58

## 3. Problem Solution
The problem is solved using the CUOPT solver, and the solution yields both the minimum objective value and the corresponding optimal variable values x. This setup demonstrates how to programmatically generate and solve a non-trivial LP using PuLP and NumPy.

In [9]:
status = prob.solve(CUOPT(msg=0))

# Print results
print("\nThe optimal value is", value(prob.objective))
x_vals = np.array([x[i].varValue for i in range(n)])
np.set_printoptions(precision=8, suppress=True)
print("A solution x is")
print(x_vals)


The optimal value is -15.22091260446782
A solution x is
[-1.08457705 -0.26613771 -0.67093999  0.14888146  0.32699776 -0.82190051
  1.30444476  1.11204819  0.62714928  0.92605598]
